In [ ]:
# Submission path setup: run notebooks from any submission subfolder.
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'Functions.ipynb').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)


In [ ]:
# !pip install "numpy<2.0"
# !pip install torch==2.2.2 torchvision==0.17.2 monai
# !pip install scikit-image
# !pip install import-ipynb
# !pip install nibabel 

In [ ]:
import os
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import monai
import torch
from tqdm.notebook import tqdm
import import_ipynb

from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from monai.transforms import (
    LoadImaged,
    Compose,
    EnsureChannelFirstd,
    Spacingd,
    ResizeWithPadOrCropd,
    NormalizeIntensityd,
    RandFlipd,
    RandRotated,
    RandGaussianSmoothd,
    Lambdad,
)

from Functions import patients_dicts, par_voxelsize


In [ ]:
def format_number(value):
    if isinstance(value, int):
        return str(value)
    if value == 0:
        return "0"
    return f"{value:g}"


def print_run_config(cfg):
    print("\n" + "=" * 100)
    print(f"RUN: {cfg['run_name']}")
    print(
        f"arch={cfg['arch_name']} | loss={cfg['loss_name']} | lr={cfg['lr']} | "
        f"weight_decay={cfg['weight_decay']} | batch_size={cfg['batch_size']} | epochs={cfg['epochs']}"
    )
    print(
        f"channels={cfg['channels']} | strides={cfg['strides']} | "
        f"min_epochs={cfg['min_epochs']} | patience={cfg['early_stopping_patience']} | "
        f"save_every={cfg['save_every_n_epochs']}"
    )
    print("=" * 100)


def compute_ascent_target_spacing(train_dataset_raw, anisotropy_threshold=3):
    spacings = []
    sizes = []
    for sample in train_dataset_raw:
        spacings.append(np.array(sample["imgED"].meta["pixdim"][1:4].tolist(), dtype=float))
        sizes.append(np.array(sample["imgED"].shape[1:4], dtype=float))

    spacings = np.vstack(spacings)
    sizes = np.vstack(sizes)

    target_spacing = np.percentile(spacings, 50, axis=0)
    target_size = np.percentile(sizes, 50, axis=0)

    worst_spacing_axis = int(np.argmax(target_spacing))
    other_axes = [i for i in range(len(target_spacing)) if i != worst_spacing_axis]
    other_spacings = [target_spacing[i] for i in other_axes]
    other_sizes = [target_size[i] for i in other_axes]

    has_aniso_spacing = target_spacing[worst_spacing_axis] > (
        anisotropy_threshold * max(other_spacings)
    )
    has_aniso_voxels = target_size[worst_spacing_axis] * anisotropy_threshold < min(other_sizes)

    if has_aniso_spacing and has_aniso_voxels:
        spacings_of_that_axis = spacings[:, worst_spacing_axis]
        target_spacing_of_that_axis = np.percentile(spacings_of_that_axis, 10)
        if target_spacing_of_that_axis < max(other_spacings):
            target_spacing_of_that_axis = (
                max(max(other_spacings), target_spacing_of_that_axis) + 1e-5
            )
        target_spacing[worst_spacing_axis] = target_spacing_of_that_axis

    return target_spacing


In [ ]:
data_path = "train"
data_path_test = "test"
runs_dir = "final_model/runs_lenient_wider2_kfold5_zspacing_only"
results_csv = os.path.join(runs_dir, "fold_results.csv")
aggregate_csv = os.path.join(runs_dir, "fold_results_aggregate.csv")

os.makedirs(runs_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"The used device is {device}")

seed = 10
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

full_dict_list = patients_dicts(data_path)
test_dict_list = patients_dicts(data_path_test)

print("train patients:", len(full_dict_list))
print("test patients:", len(test_dict_list))

train_dataset_raw = monai.data.Dataset(
    full_dict_list,
    transform=Compose([
        LoadImaged(keys=["imgED", "maskED", "imgES", "maskES"], image_only=False),
        EnsureChannelFirstd(keys=["imgED", "maskED", "imgES", "maskES"], channel_dim="no_channel"),
    ]),
)

voxel_size = []
mean_voxel, std_voxel, max_voxelsize = par_voxelsize(voxel_size, train_dataset_raw)
target_spacing = compute_ascent_target_spacing(train_dataset_raw)
print("Median voxel size:", mean_voxel)
print("ASCENT-style target spacing:", target_spacing)

voxel_volume = float(target_spacing[0] * target_spacing[1] * target_spacing[2])
roi_size = [256, 256, 16]

cfg = {
    "selection_order": 20,
    "arch_name": "wider2",
    "channels": (64, 128, 256, 512, 1024),
    "strides": (2, 2, 2, 2),
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "batch_size": 2,
    "epochs": 1000,
    "loss_name": "DiceLoss",
    "min_epochs": 400,
    "early_stopping_patience": 200,
    "early_stopping_min_delta": 1e-4,
    "save_every_n_epochs": 10,
    "num_workers": 8,
}
cfg["run_name"] = (
    f'kfold5_zspacing_only_wider2_{cfg["loss_name"]}_'
    f'lr{format_number(cfg["lr"])}_wd{format_number(cfg["weight_decay"])}_'
    f'bs{cfg["batch_size"]}_ep{cfg["epochs"]}_'
    f'min{cfg["min_epochs"]}_pat{cfg["early_stopping_patience"]}'
)

print_run_config(cfg)
print("Sliding window roi size:", roi_size)


In [ ]:
val_data_transform = Compose([
    LoadImaged(keys=["imgED", "maskED", "imgES", "maskES"], image_only=False),
    EnsureChannelFirstd(keys=["imgED", "maskED", "imgES", "maskES"], channel_dim="no_channel"),
    Spacingd(
        keys=["imgED", "maskED", "imgES", "maskES"],
        pixdim=(target_spacing[0], target_spacing[1], target_spacing[2]),
        mode=("bilinear", "nearest", "bilinear", "nearest"),
        ensure_same_shape=True,
        align_corners=False,
    ),
    ResizeWithPadOrCropd(keys=["imgED", "maskED", "imgES", "maskES"], spatial_size=tuple(roi_size)), #added cause the images needed to be devided by 16 for the strides in the Unet image
    NormalizeIntensityd(keys=["imgED", "imgES"], nonzero=True, channel_wise=True),
])

train_data_transform = Compose([
    LoadImaged(keys=["imgED", "maskED", "imgES", "maskES"], image_only=False),
    EnsureChannelFirstd(keys=["imgED", "maskED", "imgES", "maskES"], channel_dim="no_channel"),
    Spacingd(
        keys=["imgED", "maskED", "imgES", "maskES"],
        pixdim=(target_spacing[0], target_spacing[1], target_spacing[2]),
        mode=("bilinear", "nearest", "bilinear", "nearest"),
        ensure_same_shape=True,
        align_corners=False,
    ),
    ResizeWithPadOrCropd(keys=["imgED", "maskED", "imgES", "maskES"], spatial_size=tuple(roi_size)), #added cause the images needed to be devided by 16 for the strides in the Unet image we can look into if we can solve this an other way
    NormalizeIntensityd(keys=["imgED", "imgES"], nonzero=True, channel_wise=True),
    RandFlipd(keys=["imgED", "maskED", "imgES", "maskES"], prob=0.33, spatial_axis=[0, 1, 2]),
    RandRotated(
        keys=["imgED", "maskED", "imgES", "maskES"],
        range_x=0.4,
        prob=0.33,
        mode=["bilinear", "nearest", "bilinear", "nearest"],
    ),
    RandGaussianSmoothd(keys=["imgED", "imgES"], prob=0.33),
])

test_data_transform = Compose([
    LoadImaged(keys=["imgED", "maskED", "imgES", "maskES"], image_only=False),
    EnsureChannelFirstd(keys=["imgED", "maskED", "imgES", "maskES"], channel_dim="no_channel"),
    Spacingd(
        keys=["imgED", "maskED", "imgES", "maskES"],
        pixdim=(target_spacing[0], target_spacing[1], target_spacing[2]),
        mode=("bilinear", "nearest", "bilinear", "nearest"),
        ensure_same_shape=True,
        align_corners=False,
    ),
    ResizeWithPadOrCropd(keys=["imgED", "maskED", "imgES", "maskES"], spatial_size=tuple(roi_size)), #added cause the images needed to be devided by 16 for the strides in the Unet image
    NormalizeIntensityd(keys=["imgED", "imgES"], nonzero=True, channel_wise=True),
])

#using the split function to split the data
y = np.array([sample["Disease"] for sample in full_dict_list])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
folds = list(skf.split(np.zeros(len(y)), y))

fold_rows = []
for fold_idx, (_, val_idx) in enumerate(folds, start=1):
    fold_y = y[val_idx]
    fold_rows.append({"fold": fold_idx, **pd.Series(fold_y).value_counts().sort_index().to_dict()})
display(pd.DataFrame(fold_rows).fillna(0).astype({"fold": int}))

label_order = sorted(pd.Series([p["Disease"] for p in full_dict_list]).unique().tolist())
disease_label_order = [label for label in label_order if label != "NOR"]


In [ ]:
#Defining the Unet model
def build_model(cfg):
    model = monai.networks.nets.UNet(
        spatial_dims=3,
        in_channels=1,
        out_channels=4, #since we want to segment 3 heart areas+ back
        channels=cfg["channels"], #checking if the chanels are correct
        strides=cfg["strides"],
        #num_res_units=2,
    ).to(device)
    return model


def build_loaders(cfg, train_dataset, val_dataset, test_dataset):
    train_loader = monai.data.DataLoader(
        train_dataset,
        batch_size=cfg["batch_size"],
        collate_fn=monai.data.pad_list_data_collate,
        num_workers=cfg["num_workers"],
    )
    val_loader = monai.data.DataLoader(
        val_dataset,
        batch_size=cfg["batch_size"],
        collate_fn=monai.data.pad_list_data_collate,
        num_workers=cfg["num_workers"],
    )
    test_loader = monai.data.DataLoader(
        test_dataset,
        batch_size=cfg["batch_size"],
        collate_fn=monai.data.pad_list_data_collate,
        num_workers=cfg["num_workers"],
    )
    return train_loader, val_loader, test_loader


def build_loss_function(cfg):
    if cfg["loss_name"] == "DiceLoss":
        return monai.losses.DiceLoss(softmax=True, to_onehot_y=True, batch=True)
    if cfg["loss_name"] == "DiceCELoss":
        return monai.losses.DiceCELoss(softmax=True, to_onehot_y=True, batch=True)
    raise ValueError(f'Unknown loss_name: {cfg["loss_name"]}')


def evaluate_model(model, loader, only_dice=False):
    model.eval()
    inferer = monai.inferers.SlidingWindowInferer(roi_size=roi_size)

    dice_metric = monai.metrics.DiceMetric(include_background=False, reduction="mean")
    dice_metric.reset()

    if not only_dice:
        hd95_metric = monai.metrics.HausdorffDistanceMetric(
            include_background=False,
            percentile=95,
            reduction="mean",
        )
        hd95_metric.reset()

    with torch.no_grad():
        for sample in loader:
            for phase in ["ED", "ES"]:
                image = sample[f"img{phase}"].to(device)
                mask = sample[f"mask{phase}"].to(device)

                output = inferer(image, network=model)
                model_mask = torch.argmax(output, dim=1)
                model_mask = torch.nn.functional.one_hot(model_mask, num_classes=4)
                model_mask = model_mask.permute(0, 4, 1, 2, 3).float()

                gt_mask = torch.nn.functional.one_hot(
                    mask.long().squeeze(1), num_classes=4
                ).permute(0, 4, 1, 2, 3).float()

                dice_metric(y_pred=model_mask, y=gt_mask)
                if not only_dice:
                    hd95_metric(y_pred=model_mask, y=gt_mask)

    dice = dice_metric.aggregate().item()
    if only_dice:
        return {"dice": float(dice), "hd95": None}

    hd95 = hd95_metric.aggregate().item()
    return {"dice": float(dice), "hd95": float(hd95)}


feature_cols = [
    "rv_ed", "myo_ed", "lv_ed",
    "rv_es", "myo_es", "lv_es",
    "rv_sv", "lv_sv", "rv_ef", "lv_ef",
    "myo_delta", "lv_rv_ratio_ed", "lv_rv_ratio_es",
    "myo_lv_ratio_ed", "myo_lv_ratio_es",
]


def volume_from_mask(mask, label, voxel_volume):
    return float((mask == label).sum() * voxel_volume)


def safe_ef(edv, esv):
    if edv <= 0:
        return np.nan
    return float((edv - esv) / edv)


def feature_row_from_masks(patient_id, disease, mask_ed, mask_es):
    rv_ed = volume_from_mask(mask_ed, 1, voxel_volume)
    myo_ed = volume_from_mask(mask_ed, 2, voxel_volume)
    lv_ed = volume_from_mask(mask_ed, 3, voxel_volume)
    rv_es = volume_from_mask(mask_es, 1, voxel_volume)
    myo_es = volume_from_mask(mask_es, 2, voxel_volume)
    lv_es = volume_from_mask(mask_es, 3, voxel_volume)

    return {
        "ID": patient_id,
        "Disease": disease,
        "rv_ed": rv_ed,
        "myo_ed": myo_ed,
        "lv_ed": lv_ed,
        "rv_es": rv_es,
        "myo_es": myo_es,
        "lv_es": lv_es,
        "rv_sv": rv_ed - rv_es,
        "lv_sv": lv_ed - lv_es,
        "rv_ef": safe_ef(rv_ed, rv_es),
        "lv_ef": safe_ef(lv_ed, lv_es),
        "myo_delta": myo_ed - myo_es,
        "lv_rv_ratio_ed": float(lv_ed / rv_ed) if rv_ed > 0 else np.nan,
        "lv_rv_ratio_es": float(lv_es / rv_es) if rv_es > 0 else np.nan,
        "myo_lv_ratio_ed": float(myo_ed / lv_ed) if lv_ed > 0 else np.nan,
        "myo_lv_ratio_es": float(myo_es / lv_es) if lv_es > 0 else np.nan,
    }


def predict_masks_for_sample(sample, model, inferer):
    with torch.no_grad():
        image_ed = sample["imgED"].unsqueeze(0).float().to(device)
        image_es = sample["imgES"].unsqueeze(0).float().to(device)
        pred_ed = torch.argmax(inferer(image_ed, network=model), dim=1).squeeze().cpu().numpy().astype(int)
        pred_es = torch.argmax(inferer(image_es, network=model), dim=1).squeeze().cpu().numpy().astype(int)
    return pred_ed, pred_es


def extract_pred_feature_table(dataset, model):
    rows = []
    inferer = monai.inferers.SlidingWindowInferer(roi_size=roi_size)
    model.eval()
    for sample in dataset:
        pred_ed, pred_es = predict_masks_for_sample(sample, model, inferer)
        rows.append(feature_row_from_masks(sample["ID"], sample["Disease"], pred_ed, pred_es))
    return pd.DataFrame(rows)


def fit_classifier(train_df):
    X_train = train_df[feature_cols].fillna(0.0)
    y_train = train_df["Disease"]
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, multi_class="auto"))
    clf.fit(X_train, y_train)
    return clf


def evaluate_classifier_on_df(clf, split_df):
    X = split_df[feature_cols].fillna(0.0)
    y_true = split_df["Disease"]
    y_pred = clf.predict(X)

    recall_per_class = recall_score(y_true, y_pred, labels=label_order, average=None, zero_division=0)
    disease_recalls = [recall_value for label, recall_value in zip(label_order, recall_per_class) if label != "NOR"]

    stats = {
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "min_disease_recall": float(np.min(disease_recalls)) if len(disease_recalls) else np.nan,
    }

    for label, recall_value in zip(label_order, recall_per_class):
        stats[f"recall_{label}"] = float(recall_value)

    return stats


def cross_validate_classifier(feature_df, n_splits=5, seed=10):
    y = feature_df["Disease"].to_numpy()
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    fold_stats = []
    for train_idx, val_idx in skf.split(np.zeros(len(y)), y):
        fold_train_df = feature_df.iloc[train_idx].reset_index(drop=True)
        fold_val_df = feature_df.iloc[val_idx].reset_index(drop=True)
        clf = fit_classifier(fold_train_df)
        fold_stats.append(evaluate_classifier_on_df(clf, fold_val_df))

    fold_stats_df = pd.DataFrame(fold_stats)
    cv_summary = {f"{col}_val": float(fold_stats_df[col].mean()) for col in fold_stats_df.columns}
    disease_recall_val_cols = [f"recall_{label}_val" for label in disease_label_order]
    if len(disease_recall_val_cols):
        cv_summary["min_disease_recall_val"] = float(min(cv_summary[col] for col in disease_recall_val_cols))
    return cv_summary


def average_loss(model, loader, loss_function):
    model.eval()
    loss_sum = 0.0
    steps = 0

    with torch.no_grad():
        for batch in loader:
            imagED = batch["imgED"].float().to(device)
            labelED = batch["maskED"].long().to(device)
            imagES = batch["imgES"].float().to(device)
            labelES = batch["maskES"].long().to(device)

            outputED = model(imagED)
            outputES = model(imagES)

            loss = (loss_function(outputED, labelED) + loss_function(outputES, labelES)) / 2
            loss_sum += loss.item()
            steps += 1

    return float(loss_sum / steps)


def train_medmnistmodel(model, train_dataloader, val_dataloader, optimizer, cfg, run_dir):
    history_rows = []
    best_train_loss = float("inf")
    best_val_loss = float("inf")
    best_epoch = None
    loss_function = build_loss_function(cfg)

    epochs_without_improvement = 0
    stopped_early = False
    stop_epoch = None

    checkpoints_dir = os.path.join(run_dir, "checkpoints")
    os.makedirs(checkpoints_dir, exist_ok=True)

    start_time = time.time()
    table_handle = None
    epoch_bar = tqdm(range(1, cfg["epochs"] + 1), desc=os.path.basename(run_dir), position=1, leave=False)

    for epoch in epoch_bar:
        model.train()
        steps = 0
        epoch_loss = 0.0

        for batch in train_dataloader: #250 steps per epoch, figure out how many steps we need, going to the data once, early stopping if patient level =15
            optimizer.zero_grad() #backropagation
            imagED = batch["imgED"].float().to(device)
            labelsED = batch["maskED"].long().to(device)
            imagES = batch["imgES"].float().to(device)
            labelsES = batch["maskES"].long().to(device)

            outputED = model(imagED)
            outputES = model(imagES)

            loss = (loss_function(outputED, labelsED) + loss_function(outputES, labelsES)) / 2
            epoch_loss += loss.item()
            loss.backward()
            optimizer.step()
            steps += 1

        avg_train_loss = epoch_loss / steps

        val_steps = 0
        val_epoch_loss = 0.0
        model.eval()
        with torch.no_grad():
            for batch in val_dataloader:
                imagED = batch["imgED"].float().to(device)
                labelED = batch["maskED"].long().to(device)
                imagES = batch["imgES"].float().to(device)
                labelES = batch["maskES"].long().to(device)

                outputED = model(imagED)
                outputES = model(imagES)

                loss = (loss_function(outputED, labelED) + loss_function(outputES, labelES)) / 2
                val_epoch_loss += loss.item()
                val_steps += 1

        avg_val_loss = val_epoch_loss / val_steps

        row = {
            "epoch": epoch,
            "loss_train": float(avg_train_loss),
            "loss_val": float(avg_val_loss),
            "elapsed_min": (time.time() - start_time) / 60.0,
        }

        improved = avg_val_loss < (best_val_loss - cfg["early_stopping_min_delta"])

        if improved:
            best_train_loss = avg_train_loss
            best_val_loss = avg_val_loss
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(model.state_dict(), os.path.join(run_dir, "best_model.pt"))
        else:
            if epoch >= cfg["min_epochs"]:
                epochs_without_improvement += 1
                if epochs_without_improvement >= cfg["early_stopping_patience"]:
                    print(
                        f"Early stopping at epoch {epoch} | "
                        f"best epoch: {best_epoch} | best val loss: {best_val_loss:.6f}"
                    )
                    stopped_early = True
                    stop_epoch = epoch
                    history_rows.append(row)
                    break

        history_rows.append(row)
        history_df = pd.DataFrame(history_rows)
        history_df.to_csv(os.path.join(run_dir, "history.csv"), index=False)

        if epoch % cfg["save_every_n_epochs"] == 0:
            torch.save(model.state_dict(), os.path.join(checkpoints_dir, f"epoch_{epoch:04d}.pt"))

        table_view = history_df[["epoch", "loss_train", "loss_val", "elapsed_min"]].tail(8)
        if table_handle is None:
            table_handle = display(table_view, display_id=True)
        else:
            table_handle.update(table_view)

        epoch_bar.set_postfix({
            "loss_train": f"{avg_train_loss:.4f}",
            "loss_val": f"{avg_val_loss:.4f}",
        })

    history_df = pd.DataFrame(history_rows)
    history_df.to_csv(os.path.join(run_dir, "history.csv"), index=False)
    torch.save(model.state_dict(), os.path.join(run_dir, "last_model.pt"))

    return history_df, best_epoch, float(best_train_loss), float(best_val_loss), stopped_early, stop_epoch


In [ ]:
rows = []
overall_bar = tqdm(list(enumerate(folds, start=1)), desc="5-fold wider2 kfold z-spacing", position=0)

for fold_idx, (train_idx, val_idx) in overall_bar:
    fold_run_dir = os.path.join(runs_dir, f"fold_{fold_idx:02d}")
    os.makedirs(fold_run_dir, exist_ok=True)

    fold_train_dict_list = [full_dict_list[i] for i in train_idx]
    fold_val_dict_list = [full_dict_list[i] for i in val_idx]

    train_dataset = monai.data.Dataset(fold_train_dict_list, transform=train_data_transform)
    val_dataset = monai.data.Dataset(fold_val_dict_list, transform=val_data_transform)
    test_dataset = monai.data.Dataset(test_dict_list, transform=test_data_transform)

    train_loader, val_loader, test_loader = build_loaders(cfg, train_dataset, val_dataset, test_dataset)

    print("\n" + "#" * 100)
    print(f"FOLD {fold_idx}")
    print("train size:", len(train_dataset), "val size:", len(val_dataset), "test size:", len(test_dataset))
    print("#" * 100)

    model = build_model(cfg)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    train_start = time.time()
    history_df, best_epoch, best_train_loss, best_val_loss, stopped_early, stop_epoch = train_medmnistmodel(
        model,
        train_loader,
        val_loader,
        optimizer,
        cfg,
        fold_run_dir,
    )
    train_time = (time.time() - train_start) / 60.0

    model.load_state_dict(torch.load(os.path.join(fold_run_dir, "best_model.pt"), map_location=device))

    loss_function = build_loss_function(cfg)
    seg_train = {"loss": average_loss(model, train_loader, loss_function), **evaluate_model(model, train_loader)}
    seg_val = {"loss": average_loss(model, val_loader, loss_function), **evaluate_model(model, val_loader)}
    seg_test = {"loss": average_loss(model, test_loader, loss_function), **evaluate_model(model, test_loader)}

    pred_train_df = extract_pred_feature_table(train_dataset, model)
    pred_val_df = extract_pred_feature_table(val_dataset, model)
    pred_test_df = extract_pred_feature_table(test_dataset, model)

    clf_val = cross_validate_classifier(pred_train_df, n_splits=5, seed=seed)
    clf = fit_classifier(pred_train_df)
    clf_train = evaluate_classifier_on_df(clf, pred_train_df)
    clf_test = evaluate_classifier_on_df(clf, pred_test_df)

    summary = {
        "fold": fold_idx,
        "run_name": cfg["run_name"],
        "arch_name": cfg["arch_name"],
        "loss_name": cfg["loss_name"],
        "lr": cfg["lr"],
        "weight_decay": cfg["weight_decay"],
        "batch_size": cfg["batch_size"],
        "channels": str(cfg["channels"]),
        "strides": str(cfg["strides"]),
        "median_spacing": str(tuple(float(x) for x in mean_voxel)),
        "target_spacing": str(tuple(float(x) for x in target_spacing)),
        "epochs": cfg["epochs"],
        "min_epochs": cfg["min_epochs"],
        "early_stopping_patience": cfg["early_stopping_patience"],
        "early_stopping_min_delta": cfg["early_stopping_min_delta"],
        "save_every_n_epochs": cfg["save_every_n_epochs"],
        "train_size": len(train_dataset),
        "val_size": len(val_dataset),
        "test_size": len(test_dataset),
        "best_epoch": best_epoch,
        "train_time": train_time,
        "best_train_loss": best_train_loss,
        "best_val_loss": best_val_loss,
        "epochs_ran": int(history_df["epoch"].iloc[-1]),
        "stopped_early": bool(stopped_early),
        "stop_epoch": stop_epoch,
        "loss_train": seg_train["loss"],
        "loss_val": seg_val["loss"],
        "loss_test": seg_test["loss"],
        "dice_train": seg_train["dice"],
        "dice_val": seg_val["dice"],
        "dice_test": seg_test["dice"],
        "hd95_train": seg_train["hd95"],
        "hd95_val": seg_val["hd95"],
        "hd95_test": seg_test["hd95"],
        "macro_f1_train": clf_train["macro_f1"],
        "macro_f1_val": clf_val["macro_f1_val"],
        "macro_f1_test": clf_test["macro_f1"],
        "min_disease_recall_train": clf_train["min_disease_recall"],
        "min_disease_recall_val": clf_val["min_disease_recall_val"],
        "min_disease_recall_test": clf_test["min_disease_recall"],
    }

    recall_label_order = disease_label_order + (["NOR"] if "NOR" in label_order else [])
    for label in recall_label_order:
        summary[f"recall_{label}_train"] = clf_train[f"recall_{label}"]
        summary[f"recall_{label}_val"] = clf_val[f"recall_{label}_val"]
        summary[f"recall_{label}_test"] = clf_test[f"recall_{label}"]

    pred_train_df.to_csv(os.path.join(fold_run_dir, "pred_train_features.csv"), index=False)
    pred_val_df.to_csv(os.path.join(fold_run_dir, "pred_val_features.csv"), index=False)
    pred_test_df.to_csv(os.path.join(fold_run_dir, "pred_test_features.csv"), index=False)
    rows.append(summary)

    with open(os.path.join(fold_run_dir, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    pd.DataFrame(rows).to_csv(results_csv, index=False)
    display(pd.DataFrame(rows))

results_df = pd.DataFrame(rows)
results_df.to_csv(results_csv, index=False)
display(results_df)


In [ ]:
results_df = pd.read_csv(results_csv)
display(results_df)

aggregate_df = pd.DataFrame([
    {
        "metric": "loss_train",
        "mean": results_df["loss_train"].mean(),
        "std": results_df["loss_train"].std(ddof=1),
    },
    {
        "metric": "loss_val",
        "mean": results_df["loss_val"].mean(),
        "std": results_df["loss_val"].std(ddof=1),
    },
    {
        "metric": "loss_test",
        "mean": results_df["loss_test"].mean(),
        "std": results_df["loss_test"].std(ddof=1),
    },
    {
        "metric": "dice_train",
        "mean": results_df["dice_train"].mean(),
        "std": results_df["dice_train"].std(ddof=1),
    },
    {
        "metric": "dice_val",
        "mean": results_df["dice_val"].mean(),
        "std": results_df["dice_val"].std(ddof=1),
    },
    {
        "metric": "dice_test",
        "mean": results_df["dice_test"].mean(),
        "std": results_df["dice_test"].std(ddof=1),
    },
    {
        "metric": "hd95_train",
        "mean": results_df["hd95_train"].mean(),
        "std": results_df["hd95_train"].std(ddof=1),
    },
    {
        "metric": "hd95_val",
        "mean": results_df["hd95_val"].mean(),
        "std": results_df["hd95_val"].std(ddof=1),
    },
    {
        "metric": "hd95_test",
        "mean": results_df["hd95_test"].mean(),
        "std": results_df["hd95_test"].std(ddof=1),
    },
    {
        "metric": "best_epoch",
        "mean": results_df["best_epoch"].mean(),
        "std": results_df["best_epoch"].std(ddof=1),
    },
    {
        "metric": "train_time",
        "mean": results_df["train_time"].mean(),
        "std": results_df["train_time"].std(ddof=1),
    },
])

aggregate_df.to_csv(aggregate_csv, index=False)
display(aggregate_df)
